## Acknowledgement of AI Assistance

Generative AI was used to assist with cleaning, reorganising and improving the presentation of this notebook. 

# Computational Model of Clock Timing Error During GNSS Holdover

## 1. Introduction

This notebook models the timing performance of a chip-scale atomic clock (CSAC) and a temperature-compensated crystal oscillator (TCXO) during GNSS holdover.

The analysis considers two contributions:

- **Ageing**, representing systematic frequency drift.
- **Stochastic frequency noise**, including white FM, flicker FM and random-walk FM.

These contributions are combined to obtain a timing-error scale, which is then translated into a range-equivalent error for underwater acoustic ranging.

## 2. Mathematical Model

Clock performance during holdover begins with the fractional frequency deviation, which describes how far the actual clock frequency differs from its nominal frequency:

$$
y(t)=\frac{f(t)-f_0}{f_0}
$$

where $f(t)$ is the clock frequency at time $t$ and $f_0$ is the nominal frequency.

A small frequency deviation produces timing error when it continues over time. The accumulated timing error is therefore

$$
x(t)=\int_0^t y(u)\,du
$$

where $x(t)$ is the accumulated timing error after a holdover duration $t$. The model assumes $x(0)=0$, meaning that the timing error is zero at the beginning of holdover.

In this model, the timing error is evaluated from two contributions: **ageing** and **stochastic frequency noise**.

## 3. Model Parameters and Data Sources

The model compares a representative **Microchip SA.45s CSAC** with an **IQD IQXT-220-1 TCXO**. Two types of input data are required because ageing and stochastic frequency noise are modelled separately.

For ageing, the SA.45s datasheet specifies a typical long-term ageing of less than $9\times10^{-10}$ per month, while the IQXT-220-1 specifies a maximum ageing of $\pm0.02$ ppm per day. These values are converted to fractional-frequency change per second for use in the ageing model.

For stochastic frequency noise, representative power-law coefficients are taken from Fernández et al. (2017). The three coefficients describe white FM ($h_0$), flicker FM ($h_{-1}$), and random-walk FM ($h_{-2}$).

| Clock | Ageing specification | $h_0$ | $h_{-1}$ | $h_{-2}$ |
|---|---:|---:|---:|---:|
| SA.45s CSAC | $<9\times10^{-10}$/month | $7.2\times10^{-21}$ | $2.6\times10^{-23}$ | $2.7\times10^{-27}$ |
| IQXT-220-1 TCXO | $\pm0.02$ ppm/day | $9.4\times10^{-20}$ | $1.8\times10^{-19}$ | $3.8\times10^{-21}$ |

The ageing values are manufacturer specifications, whereas the stochastic coefficients are generic theoretical values reported under unspecified conditions. Therefore, the stochastic coefficients are used as representative parameters rather than device-specific measurements of the SA.45s and IQXT-220-1.

The power-law coefficients are converted to Allan variance using the relationships in NBS Monograph 140. The corresponding Allan deviations are then converted to RMS time-prediction uncertainty using the relationships in NBS Technical Note 1337.

In [16]:
import numpy as np
import pandas as pd

## 4. Ageing Contribution to Timing Error

Ageing represents a gradual systematic change in the clock frequency over time. For the short holdover periods considered in this model, the ageing specification is approximated as a constant linear fractional-frequency drift rate, $D$.

The fractional-frequency contribution from ageing is therefore

$$
y_{\mathrm{age}}(t)=Dt.
$$

Timing error is the accumulation of fractional-frequency deviation over time:

$$
x(t)=\int_0^t y(u)\,du.
$$

Substituting the linear ageing model gives

$$
x_{\mathrm{age}}(t)
=
\int_0^t Du\,du
=
\frac{1}{2}Dt^2.
$$

The manufacturer ageing specifications are first converted to fractional-frequency change per second. The resulting values of $D$ are then used to calculate the ageing contribution at each holdover duration.

In [17]:

seconds_per_day = 24 * 60 * 60
seconds_per_month = 30 * seconds_per_day

D_csac = 9e-10 / seconds_per_month
D_tcxo = 2e-8 / seconds_per_day

print(f"CSAC ageing rate: {D_csac:.3e} s^-1")
print(f"TCXO ageing rate: {D_tcxo:.3e} s^-1")

CSAC ageing rate: 3.472e-16 s^-1
TCXO ageing rate: 2.315e-13 s^-1


The ageing timing error is calculated from

$$
x_{\mathrm{age}}(t)=\frac{1}{2}Dt^2.
$$

The calculation is performed for the four baseline holdover durations: 10 min, 1 h, 6 h and 12 h.

In [18]:
def ageing_timing_error(D, t):
    """Calculate timing error caused by constant linear ageing."""
    return 0.5 * D * t**2


holdover_times = {
    "10 min": 600,
    "1 h": 3600,
    "6 h": 21600,
    "12 h": 43200,
}

ageing_results = []

for label, t in holdover_times.items():

    csac_age_s = ageing_timing_error(D_csac, t)
    tcxo_age_s = ageing_timing_error(D_tcxo, t)

    ageing_results.append({
        "Holdover": label,
        "CSAC ageing error (µs)": csac_age_s * 1e6,
        "TCXO ageing error (µs)": tcxo_age_s * 1e6,
    })

ageing_df = pd.DataFrame(ageing_results)
ageing_df

,Holdover,CSAC ageing error (µs),TCXO ageing error (µs)
0,10 min,0.000062,0.041667
1,1 h,0.002250,1.500000
2,6 h,0.081000,54.000000
3,12 h,0.324000,216.000000


The ageing contribution increases with the square of holdover time because the assumed frequency drift accumulates continuously. Under this model, the TCXO develops a much larger ageing-related timing error than the CSAC as the holdover duration increases.

These results represent the ageing contribution only. Stochastic frequency fluctuations are evaluated separately in the following section.

## 5. Stochastic Timing Uncertainty

In addition to ageing, random frequency fluctuations contribute to timing uncertainty during holdover. The stochastic model includes three frequency-noise processes: **white FM**, **flicker FM**, and **random-walk FM**.

The representative noise coefficients $h_0$, $h_{-1}$, and $h_{-2}$ given in Section 3 describe these processes in the frequency domain.

Using the standard power-law relationships in NBS Monograph 140, the corresponding Allan variances are

$$
\sigma_{y,W}^2(\tau)=\frac{h_0}{2\tau},
$$

$$
\sigma_{y,F}^2(\tau)=2\ln(2)h_{-1},
$$

and

$$
\sigma_{y,RW}^2(\tau)=\frac{2\pi^2}{3}h_{-2}\tau,
$$

where $\tau$ is the averaging time.

Taking the square root gives the Allan deviation associated with each noise process.

### 5.1 From Frequency Instability to Timing Uncertainty

The Allan deviations describe frequency instability rather than timing uncertainty. To estimate the corresponding time-prediction uncertainty, the relationships given in NBS Technical Note 1337 are applied to each noise process separately.

For a prediction interval $\tau_p$:

$$
\sigma_{x,W}(\tau_p)
=
\tau_p\,\sigma_{y,W}(\tau_p),
$$

$$
\sigma_{x,F}(\tau_p)
=
\frac{\tau_p\,\sigma_{y,F}(\tau_p)}
{\sqrt{\ln(2)}},
$$

and

$$
\sigma_{x,RW}(\tau_p)
=
\tau_p\,\sigma_{y,RW}(\tau_p).
$$

Here, $\tau_p$ is taken as the holdover duration being evaluated. The resulting quantities represent RMS time-prediction uncertainty due to each stochastic noise process.

Assuming the three stochastic contributions are independent, their variances are combined as

$$
\sigma_{x,\mathrm{stoch}}
=
\sqrt{
\sigma_{x,W}^2+
\sigma_{x,F}^2+
\sigma_{x,RW}^2
}.
$$

In [19]:
csac_noise = {
    "h0": 7.2e-21,
    "h_minus1": 2.6e-23,
    "h_minus2": 2.7e-27,
}

tcxo_noise = {
    "h0": 9.4e-20,
    "h_minus1": 1.8e-19,
    "h_minus2": 3.8e-21,
}

In [ ]:

def stochastic_timing_uncertainty(t, noise):
    """
    Calculate RMS stochastic timing uncertainty from
    white FM, flicker FM and random-walk FM.
    """

    h0 = noise["h0"]
    h_minus1 = noise["h_minus1"]
    h_minus2 = noise["h_minus2"]

    # Allan variance of each noise process
    var_white = h0 / (2 * t)
    var_flicker = 2 * np.log(2) * h_minus1
    var_random_walk = (2 * np.pi**2 / 3) * h_minus2 * t

    # Allan deviation
    sigma_y_white = np.sqrt(var_white)
    sigma_y_flicker = np.sqrt(var_flicker)
    sigma_y_random_walk = np.sqrt(var_random_walk)

    # time-prediction uncertainty
    sigma_x_white = t * sigma_y_white
    sigma_x_flicker = (
        t * sigma_y_flicker / np.sqrt(np.log(2))
    )
    sigma_x_random_walk = t * sigma_y_random_walk

    # Combine independent stochastic contributions
    sigma_x_stoch = np.sqrt(
        sigma_x_white**2
        + sigma_x_flicker**2
        + sigma_x_random_walk**2
    )

    return {
        "white": sigma_x_white,
        "flicker": sigma_x_flicker,
        "random_walk": sigma_x_random_walk,
        "total": sigma_x_stoch,
    }

In [21]:
stochastic_results = []

for label, t in holdover_times.items():

    csac = stochastic_timing_uncertainty(t, csac_noise)
    tcxo = stochastic_timing_uncertainty(t, tcxo_noise)

    stochastic_results.append({
        "Holdover": label,
        "CSAC stochastic uncertainty (µs)": csac["total"] * 1e6,
        "TCXO stochastic uncertainty (µs)": tcxo["total"] * 1e6,
    })

stochastic_df = pd.DataFrame(stochastic_results)
stochastic_df

,Holdover,CSAC stochastic uncertainty (µs),TCXO stochastic uncertainty (µs)
0,10 min,0.004972,2.351654
1,1 h,0.038932,34.222882
2,6 h,0.450968,502.136011
3,12 h,1.236716,1420.018569


## 6. Baseline Holdover Comparison

The ageing and stochastic contributions represent different aspects of clock behaviour. Ageing is treated as a systematic timing contribution, while the stochastic component represents RMS time-prediction uncertainty.

To express both contributions as a single timing-error scale, the stochastic contribution is treated as zero-mean and combined with the ageing contribution using

$$
x_{\mathrm{RMS}}(t)
=
\sqrt{
x_{\mathrm{age}}^2(t)
+
\sigma_{x,\mathrm{stoch}}^2(t)
}.
$$

This quantity is used as a **timing-error scale** for comparing the CSAC and TCXO during holdover. It should not be interpreted as the exact timing error of a specific clock.

In [22]:
baseline_results = []

for label, t in holdover_times.items():

    # Ageing 
    csac_age = ageing_timing_error(D_csac, t)
    tcxo_age = ageing_timing_error(D_tcxo, t)


    # Stochastic 
    csac_stoch = stochastic_timing_uncertainty(
        t, csac_noise
    )["total"]

    tcxo_stoch = stochastic_timing_uncertainty(
        t, tcxo_noise
    )["total"]


    # Combined timing-error 

    csac_rms = np.sqrt(
        csac_age**2 + csac_stoch**2
    )

    tcxo_rms = np.sqrt(
        tcxo_age**2 + tcxo_stoch**2
    )


    baseline_results.append({
        "Holdover": label,

        "CSAC ageing (µs)":
            csac_age * 1e6,

        "CSAC stochastic (µs)":
            csac_stoch * 1e6,

        "CSAC timing-error scale (µs)":
            csac_rms * 1e6,

        "TCXO ageing (µs)":
            tcxo_age * 1e6,

        "TCXO stochastic (µs)":
            tcxo_stoch * 1e6,

        "TCXO timing-error scale (µs)":
            tcxo_rms * 1e6,
    })


baseline_df = pd.DataFrame(baseline_results)

baseline_df.round(6)

,Holdover,CSAC ageing (µs),CSAC stochastic (µs),CSAC timing-error scale (µs),TCXO ageing (µs),TCXO stochastic (µs),TCXO timing-error scale (µs)
0,10 min,0.000062,0.004972,0.004972,0.041667,2.351654,2.352023
1,1 h,0.002250,0.038932,0.038997,1.500000,34.222882,34.255739
2,6 h,0.081000,0.450968,0.458185,54.000000,502.136011,505.031260
3,12 h,0.324000,1.236716,1.278453,216.000000,1420.018569,1436.352581


The combined timing-error scale increases with holdover duration for both clocks. However, the increase is much smaller for the CSAC. After 12 h, the CSAC timing-error scale is approximately $1.28~\mu\mathrm{s}$, compared with approximately $1.44~\mathrm{ms}$ for the TCXO.

The stochastic contribution remains larger than the ageing contribution for both clocks over the baseline holdover periods considered here.\\


## 7. Comparison with Experimental CSAC Holdover Results

The baseline analysis is limited to 12 h. However, published experimental data provide an opportunity to examine whether the model produces a reasonable timing-error scale over a longer holdover period.

Kim and Walter (2017) experimentally evaluated multiple CSAC units after GPS disciplining. With a 30 s disciplining time constant, all tested units except one remained within approximately $20~\mu\mathrm{s}$ of clock error over three days at room temperature.

For comparison, the CSAC model is extended to 24, 48 and 72 h using the same ageing and stochastic-noise assumptions applied in the baseline analysis.

In [23]:

extended_csac_times = {
    "12 h": 12 * 60 * 60,
    "24 h": 24 * 60 * 60,
    "48 h": 48 * 60 * 60,
    "72 h": 72 * 60 * 60,
}

extended_csac_results = []

for label, t in extended_csac_times.items():

    # Ageing 
    csac_age = ageing_timing_error(
        D_csac, t
    )

    # Stochastic
    csac_stoch = stochastic_timing_uncertainty(
        t, csac_noise
    )["total"]

    # Combined timing-error 
    csac_rms = np.sqrt(
        csac_age**2 + csac_stoch**2
    )

    extended_csac_results.append({
        "Holdover": label,
        "Ageing (µs)": csac_age * 1e6,
        "Stochastic uncertainty (µs)": csac_stoch * 1e6,
        "Timing-error scale (µs)": csac_rms * 1e6,
    })

extended_csac_df = pd.DataFrame(
    extended_csac_results
)

extended_csac_df.round(3)

,Holdover,Ageing (µs),Stochastic uncertainty (µs),Timing-error scale (µs)
0,12 h,0.324,1.237,1.278
1,24 h,1.296,3.442,3.678
2,48 h,5.184,9.655,10.959
3,72 h,11.664,17.688,21.188


At 72 h, the model gives a CSAC timing-error scale of approximately $21.2~\mu\mathrm{s}$. This is comparable in scale to the approximately $20~\mu\mathrm{s}$ experimental bound reported by Kim and Walter (2017) for most of the tested CSAC units.

Although the values are similar, the comparison only shows that the model produces a timing-error scale consistent with published experimental results; it does not prove that the model can predict the exact behaviour of a specific CSAC.

## 8. Underwater Range-Equivalent Error

One-way acoustic ranging determines distance from the travel time of an acoustic signal. The basic relationship is

$$
R=c_s t,
$$

where $R$ is range, $c_s$ is the acoustic propagation speed, and $t$ is the measured travel time.

A clock timing error $\Delta t$ therefore produces a corresponding range-equivalent error

$$
\Delta R=c_s\Delta t.
$$

An effective acoustic propagation speed of $1481~\mathrm{m/s}$ is used, based on the experimental acoustic calibration reported by Rypkema et al. (2022).

The conversion is applied to the combined timing-error scales of both clocks. The baseline durations are extended to 24, 48 and 72 h to show how the clock-related ranging contribution develops over longer holdover periods.

In [24]:

# acoustic propagation speed from Rypkema et al. (2022)
sound_speed = 1481.0  # m/s


# Holdover durations
range_times = {
    "10 min": 10 * 60,
    "1 h": 1 * 60 * 60,
    "6 h": 6 * 60 * 60,
    "12 h": 12 * 60 * 60,
    "24 h": 24 * 60 * 60,
    "48 h": 48 * 60 * 60,
    "72 h": 72 * 60 * 60,
}


range_results = []

for label, t in range_times.items():

    # CSAC

    csac_age = ageing_timing_error(
        D_csac, t
    )

    csac_stoch = stochastic_timing_uncertainty(
        t, csac_noise
    )["total"]

    csac_rms = np.sqrt(
        csac_age**2 + csac_stoch**2
    )

    csac_range = sound_speed * csac_rms


    # TCXO

    tcxo_age = ageing_timing_error(
        D_tcxo, t
    )

    tcxo_stoch = stochastic_timing_uncertainty(
        t, tcxo_noise
    )["total"]

    tcxo_rms = np.sqrt(
        tcxo_age**2 + tcxo_stoch**2
    )

    tcxo_range = sound_speed * tcxo_rms


    
    range_results.append({
        "Holdover": label,
        "CSAC timing-error scale (µs)": csac_rms * 1e6,
        "CSAC range-equivalent error (m)": csac_range,
        "TCXO timing-error scale (µs)": tcxo_rms * 1e6,
        "TCXO range-equivalent error (m)": tcxo_range,
    })


range_df = pd.DataFrame(range_results)

range_df.round({
    "CSAC timing-error scale (µs)": 3,
    "CSAC range-equivalent error (m)": 6,
    "TCXO timing-error scale (µs)": 3,
    "TCXO range-equivalent error (m)": 3,
})

,Holdover,CSAC timing-error scale (µs),CSAC range-equivalent error (m),TCXO timing-error scale (µs),TCXO range-equivalent error (m)
0,10 min,0.005,0.000007,2.352,0.003
1,1 h,0.039,0.000058,34.256,0.051
2,6 h,0.458,0.000679,505.031,0.748
3,12 h,1.278,0.001893,1436.353,2.127
4,24 h,3.678,0.005447,4107.972,6.084
5,48 h,10.959,0.016230,11872.854,17.584
6,72 h,21.188,0.031379,22268.791,32.980


The range-equivalent contribution increases with holdover duration, but at very different rates for the two clocks. For the CSAC, the clock-related range-equivalent error remains below approximately $2~\mathrm{mm}$ after 12 h and increases to approximately $3.1~\mathrm{cm}$ after 72 h. In comparison, the TCXO contribution reaches approximately $2.1~\mathrm{m}$ after 12 h and approximately $33~\mathrm{m}$ after 72 h.

The 72 h CSAC result is also comparable in scale to experimental results reported by Kebkal et al. (2017), where CSAC timing deviations below approximately $20~\mu\mathrm{s}$ corresponded to clock-related ranging errors below approximately $3~\mathrm{cm}$.

These values represent only the range-equivalent contribution associated with clock timing. Actual underwater acoustic ranging is also affected by factors such as sound-speed variation and other measurement uncertainties. The results should therefore not be interpreted as the total ranging error or overall navigation accuracy of the AUV.